In [177]:
import pandas as pd

real = pd.read_csv("../data/Real_Estate_Data.csv")
rent = pd.read_csv("../data/House_Rent_Dataset.csv")
apt = pd.read_csv("../data/apartment.csv")
crime = pd.read_csv("../data/Crime_Rate.csv")
weather = pd.read_csv("../data/Weather_Events_India.csv")


In [178]:
real.head()
# rent.head()
# apt.head()
# crime.head()
# weather.head()


,Name,Property Title,Price,Location,Total_Area,Price_per_SQFT,Description,Baths,Balcony
0,Casagrand ECR 14,"4 BHK Flat for sale in Kanathur Reddikuppam, C...",₹1.99 Cr,"Kanathur Reddikuppam, Chennai",2583,7700.0,Best 4 BHK Apartment for modern-day lifestyle ...,4,Yes
1,"Ramanathan Nagar, Pozhichalur,Chennai",10 BHK Independent House for sale in Pozhichal...,₹2.25 Cr,"Ramanathan Nagar, Pozhichalur,Chennai",7000,3210.0,Looking for a 10 BHK Independent House for sal...,6,Yes
2,DAC Prapthi,"3 BHK Flat for sale in West Tambaram, Chennai",₹1.0 Cr,"Kasthuribai Nagar, West Tambaram,Chennai",1320,7580.0,"Property for sale in Tambaram, Chennai. This 3...",3,No
3,"Naveenilaya,Chepauk, Triplicane,Chennai",7 BHK Independent House for sale in Triplicane...,₹3.33 Cr,"Naveenilaya,Chepauk, Triplicane,Chennai",4250,7840.0,Entire Building for sale with 7 units of singl...,5,Yes
4,VGN Spring Field Phase 1,"2 BHK Flat for sale in Avadi, Chennai",₹48.0 L,"Avadi, Chennai",960,5000.0,"Property for sale in Avadi, Chennai. This 2 BH...",3,Yes


In [179]:
import pandas as pd
import numpy as np
import re

# ==========================
# 1. LOAD ALL DATASETS
# ==========================
real   = pd.read_csv("../data/Real_Estate_Data.csv")
rent   = pd.read_csv("../data/House_Rent_Dataset.csv")
crime  = pd.read_csv("../data/Crime_Rate.csv")
# apt   = pd.read_csv("../data/apartment.csv")   # not used in this pipeline
# weather = pd.read_csv("../data/Weather_Events_India.csv")  # for later (flood risk)

print("Loaded shapes:")
print("real :", real.shape)
print("rent :", rent.shape)
print("crime:", crime.shape)


# ==========================
# 2. PRICE CLEANING FUNCTION (your function)
# ==========================
def clean_price(x):
    if pd.isna(x):
        return np.nan
    
    s = str(x).lower()
    s = s.replace("₹", "").replace(",", "").strip()
    
    # ignore weird units like acres
    if "acs" in s or "acre" in s:
        return np.nan
    
    multiplier = 1.0
    
    # crore
    if "cr" in s:
        multiplier = 10000000     # 1 Cr
        s = s.replace("cr", "")
    # lakh / lac / L
    elif "lac" in s:
        multiplier = 100000
        s = s.replace("lac", "")
    elif "l" in s:
        multiplier = 100000
        s = s.replace("l", "")
    
    nums = re.findall(r"[\d\.]+", s)
    if not nums:
        return np.nan
    
    try:
        return float(nums[0]) * multiplier
    except:
        return np.nan


# ==========================
# 3. CLEAN REAL ESTATE → real_clean
# ==========================
# Split Location into Locality + City
real["Location"] = real["Location"].astype(str)
loc_split = real["Location"].str.split(",", n=1, expand=True)

real["Locality"] = loc_split[0].astype(str).str.strip()
real["City"] = loc_split[1].astype(str).fillna("").str.strip()

# Clean price
real["Price_clean"] = real["Price"].apply(clean_price)

# Build cleaned property table
real_clean = real[[
    "Name",
    "Property Title",
    "Price_clean",
    "Total_Area",
    "Locality",
    "City"
]].copy()

real_clean.rename(columns={
    "Price_clean": "Price",
    "Total_Area": "Area_sqft"
}, inplace=True)

real_clean["City"] = real_clean["City"].astype(str).str.strip()
real_clean["Locality"] = real_clean["Locality"].astype(str).str.strip()

print("\nreal_clean sample:")
display(real_clean.head(5))


# ==========================
# 4. CLEAN RENT DATA → rent_grouped
# ==========================
rent.rename(columns={"Area Locality": "Locality"}, inplace=True)

rent["City"] = rent["City"].astype(str).str.strip()
rent["Locality"] = rent["Locality"].astype(str).str.strip()

rent_grouped = rent.groupby(["City","Locality"])["Rent"].mean().reset_index()
rent_grouped.rename(columns={"Rent": "Avg_Rent"}, inplace=True)

print("\nrent_grouped sample:")
display(rent_grouped.head(5))


# ==========================
# 5. CLEAN CRIME DATA → crime_small
# ==========================
crime.columns = crime.columns.str.strip()

crime_small = crime[[
    "City",
    "Total   - Total Persons Arrested by age and Sex"
]].copy()

crime_small.rename(
    columns={"Total   - Total Persons Arrested by age and Sex": "Crime_Index"},
    inplace=True
)

crime_small["City"] = crime_small["City"].astype(str).str.strip()

print("\ncrime_small sample:")
display(crime_small.head(5))


# ==========================
# 6. MERGE: real_clean + rent_grouped + crime_small → df
# ==========================
df = real_clean.merge(
    rent_grouped,
    on=["City","Locality"],
    how="left"
)

# Drop old Crime_Index if present
df = df.drop(columns=["Crime_Index"], errors="ignore")

df = df.merge(
    crime_small,
    on="City",
    how="left"
)

print("\nMerged df sample (after rent + crime):")
display(df[["Name","City","Locality","Price","Avg_Rent","Crime_Index"]].head(10))


# ==========================
# 7. CLEAN Crime_Index (simple & robust)
# ==========================
df["Crime_Index"] = pd.to_numeric(df["Crime_Index"], errors="coerce")

if df["Crime_Index"].notna().sum() == 0:
    # no valid data at all → assume 0
    df["Crime_Index"] = 0.0
else:
    # fill NAs with median of available values
    median_ci = float(df["Crime_Index"].median())
    df["Crime_Index"] = df["Crime_Index"].fillna(median_ci)

print("\nCrime_Index after cleaning:")
print("Missing:", df["Crime_Index"].isna().sum())
print(df["Crime_Index"].describe())


# ==========================
# 8. RENTAL YIELD + INVESTMENT SCORE
# ==========================
# rental yield (%)
df["rental_yield_pct"] = (df["Avg_Rent"] * 12) / df["Price"] * 100
df["rental_yield_pct"] = df["rental_yield_pct"].replace([np.inf, -np.inf], np.nan).fillna(0)

def normalize(series):
    s = series.astype(float)
    smin, smax = s.min(), s.max()
    if smin == smax:
        return s*0
    return (s - smin) / (smax - smin)

df["ry_norm"] = normalize(df["rental_yield_pct"])
df["crime_norm"] = normalize(df["Crime_Index"])
df["safety_norm"] = 1 - df["crime_norm"]

# final investment score (0–100)
df["investment_score"] = (
    0.7 * df["ry_norm"] +
    0.3 * df["safety_norm"]
) * 100

df["investment_score"] = df["investment_score"].clip(0,100).round(1)

print("\nFinal sample with investment_score:")
display(df[[
    "Name","City","Locality","Price","Avg_Rent",
    "rental_yield_pct","Crime_Index","investment_score"
]].head(15))


# ==========================
# 9. SAVE FINAL MASTER DATASET
# ==========================
df.to_csv("../data/master_dataset_final.csv", index=False)
print("\n✅ Saved ../data/master_dataset_final.csv")


Loaded shapes:
real : (14528, 9)
rent : (4746, 12)
crime: (20, 20)

real_clean sample:


,Name,Property Title,Price,Area_sqft,Locality,City
0,Casagrand ECR 14,"4 BHK Flat for sale in Kanathur Reddikuppam, C...",19900000.0,2583,Kanathur Reddikuppam,Chennai
1,"Ramanathan Nagar, Pozhichalur,Chennai",10 BHK Independent House for sale in Pozhichal...,22500000.0,7000,Ramanathan Nagar,"Pozhichalur,Chennai"
2,DAC Prapthi,"3 BHK Flat for sale in West Tambaram, Chennai",10000000.0,1320,Kasthuribai Nagar,"West Tambaram,Chennai"
3,"Naveenilaya,Chepauk, Triplicane,Chennai",7 BHK Independent House for sale in Triplicane...,33300000.0,4250,Naveenilaya,"Chepauk, Triplicane,Chennai"
4,VGN Spring Field Phase 1,"2 BHK Flat for sale in Avadi, Chennai",4800000.0,960,Avadi,Chennai



rent_grouped sample:


,City,Locality,Avg_Rent
0,Bangalore,"A Narayanapura, Mahadevapura",14500.000000
1,Bangalore,Aarna Enclave,90000.000000
2,Bangalore,"Abbiareddy Layout, Kaggadasapura",22500.000000
3,Bangalore,Abbigere,9333.333333
4,Bangalore,"Aditya Nagar-Vidyaranyapura, Vidyaranyapura",10000.000000



crime_small sample:


...


Merged df sample (after rent + crime):


,Name,City,Locality,Price,Avg_Rent,Crime_Index
0,Casagrand ECR 14,Chennai,Kanathur Reddikuppam,19900000.0,14000.000000,NaN
1,"Ramanathan Nagar, Pozhichalur,Chennai","Pozhichalur,Chennai",Ramanathan Nagar,22500000.0,NaN,NaN
2,DAC Prapthi,"West Tambaram,Chennai",Kasthuribai Nagar,10000000.0,NaN,NaN
3,"Naveenilaya,Chepauk, Triplicane,Chennai","Chepauk, Triplicane,Chennai",Naveenilaya,33300000.0,NaN,NaN
4,VGN Spring Field Phase 1,Chennai,Avadi,4800000.0,8000.000000,NaN
5,KG Earth Homes,Chennai,Siruseri,4000000.0,18833.333333,NaN
6,"THIRAN FLATS ,Gowrivakkam, Sembakkam,Chennai","Gowrivakkam, Sembakkam,Chennai",THIRAN FLATS,6000000.0,NaN,NaN
7,TK Jasmine Grove,Chennai,Mahindra World City,7235000.0,13333.333333,NaN
8,Avenue,"West Tambaram,Chennai",Brindavan Colony,4200000.0,NaN,NaN
9,Guru Kothai Aparts,"Chromepet,Chennai",New Colony,3000000.0,NaN,NaN



Crime_Index after cleaning:
Missing: 0
count    14528.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: Crime_Index, dtype: float64

Final sample with investment_score:


,Name,City,Locality,Price,Avg_Rent,rental_yield_pct,Crime_Index,investment_score
0,Casagrand ECR 14,Chennai,Kanathur Reddikuppam,19900000.0,14000.000000,0.844221,0.0,30.0
1,"Ramanathan Nagar, Pozhichalur,Chennai","Pozhichalur,Chennai",Ramanathan Nagar,22500000.0,NaN,0.000000,0.0,30.0
2,DAC Prapthi,"West Tambaram,Chennai",Kasthuribai Nagar,10000000.0,NaN,0.000000,0.0,30.0
3,"Naveenilaya,Chepauk, Triplicane,Chennai","Chepauk, Triplicane,Chennai",Naveenilaya,33300000.0,NaN,0.000000,0.0,30.0
4,VGN Spring Field Phase 1,Chennai,Avadi,4800000.0,8000.000000,2.000000,0.0,30.0
5,KG Earth Homes,Chennai,Siruseri,4000000.0,18833.333333,5.650000,0.0,30.0
6,"THIRAN FLATS ,Gowrivakkam, Sembakkam,Chennai","Gowrivakkam, Sembakkam,Chennai",THIRAN FLATS,6000000.0,NaN,0.000000,0.0,30.0
7,TK Jasmine Grove,Chennai,Mahindra World City,7235000.0,13333.333333,2.211472,0.0,30.0
8,Avenue,"West Tambaram,Chennai",Brindavan Colony,4200000.0,NaN,0.000000,0.0,30.0
9,Guru Kothai Aparts,"Chromepet,Chennai",New Colony,3000000.0,NaN,0.000000,0.0,30.0



✅ Saved ../data/master_dataset_final.csv


In [180]:
real[real["Price"].astype(str).str.contains("acs", case=False, na=False)][["Price", "Property Title"]].head()


,Price,Property Title
4757,₹95.45 Lacs,"3 BHK Flat for sale in Bommanahalli, Bangalore"


In [181]:
# ---------- Robust city cleaning, re-merge crime, recompute scores ----------

import pandas as pd
import numpy as np
import re

# 1) Robust city extraction for df: take last non-empty chunk after comma
def extract_main_city(x):
    try:
        s = str(x)
    except:
        return ""
    # if empty or 'nan' return empty string
    if s.strip().lower() in ("", "nan", "none"):
        return ""
    parts = [p.strip() for p in s.split(",") if p is not None and str(p).strip() != ""]
    if len(parts) == 0:
        return ""
    return parts[-1]

df["City_clean"] = df["City"].apply(extract_main_city)

# 2) Robust city extraction for crime_small:
#    extract text BEFORE any '(' or '[' or '-' or ',' (stop at common separators)
def extract_city_from_crime(s):
    try:
        s = str(s)
    except:
        return ""
    if s.strip().lower() in ("", "nan", "none"):
        return ""
    # split on first '(' or '[' or '-' or ',' and take left side
    m = re.split(r"[\(\[\-,]", s, maxsplit=1)
    city = m[0].strip()
    return city

crime_small["City_clean"] = crime_small["City"].apply(extract_city_from_crime)

# 3) Normalize common name variants (expand map if you know others)
city_map = {
    "Bengaluru": "Bangalore",
    "BengaluruKarnataka": "Bangalore",
    "Bengaluru (Karnataka)": "Bangalore",
    "Bengaluru(Karnataka)": "Bangalore",
    "Bengaluru , Karnataka": "Bangalore",
    "Mumbai": "Mumbai",
    "Bombay": "Mumbai",
    "ChennaiTamil Nadu": "Chennai",
    "Chennai Tamil Nadu": "Chennai",
    "Chennai(Tamil Nadu)": "Chennai"
    # add more mappings if you discover them
}

# apply mapping to both
df["City_clean"] = df["City_clean"].replace(city_map)
crime_small["City_clean"] = crime_small["City_clean"].replace(city_map)

# 4) trim whitespace
df["City_clean"] = df["City_clean"].astype(str).str.strip()
crime_small["City_clean"] = crime_small["City_clean"].astype(str).str.strip()

# quick sanity check: show unique city samples (first 15)
print("Sample df City -> City_clean")
display(df[["City","City_clean"]].head(15))
print("Sample crime_small City -> City_clean -> Crime_Index")
display(crime_small[["City","City_clean","Crime_Index"]].head(15))

# 5) aggregate crime by cleaned city (use mean to collapse duplicates)
crime_city = crime_small.groupby("City_clean", as_index=False)["Crime_Index"].mean()

print("crime_city sample:")
display(crime_city.head(15))

# 6) drop old Crime_Index in df and merge the new one
df = df.drop(columns=["Crime_Index"], errors="ignore")
df = df.merge(crime_city, on="City_clean", how="left")

# 7) Fill missing crime: if there are no valid numbers anywhere, set 0; otherwise use city median
df["Crime_Index"] = pd.to_numeric(df["Crime_Index"], errors="coerce")
valid_count = df["Crime_Index"].notna().sum()
if valid_count == 0:
    df["Crime_Index"] = 0.0
else:
    df["Crime_Index"] = df["Crime_Index"].fillna(df["Crime_Index"].median())

print("After merge - Crime_Index count / stats:")
print("Valid (non-NA) count:", int(df['Crime_Index'].notna().sum()))
display(df["Crime_Index"].describe())

# 8) Recompute rental_yield_pct if not already present
if "rental_yield_pct" not in df.columns:
    df["rental_yield_pct"] = (df["Avg_Rent"] * 12) / df["Price"] * 100
    df["rental_yield_pct"] = df["rental_yield_pct"].replace([np.inf, -np.inf], np.nan).fillna(0)

# 9) normalize & investment score
def normalize(s):
    s = s.astype(float)
    smin, smax = s.min(), s.max()
    if pd.isna(smin) or pd.isna(smax) or smin == smax:
        return s * 0
    return (s - smin) / (smax - smin)

df["ry_norm"] = normalize(df["rental_yield_pct"])
df["crime_norm"] = normalize(df["Crime_Index"])
df["safety_norm"] = 1 - df["crime_norm"]

# re-weight as you had before (70% ry, 30% safety)
df["investment_score"] = (0.7 * df["ry_norm"] + 0.3 * df["safety_norm"]) * 100
df["investment_score"] = df["investment_score"].clip(0,100).round(1)

# 10) show verification table
display_cols = [
    "Name","City","City_clean","Locality","Price","Avg_Rent",
    "rental_yield_pct","Crime_Index","ry_norm","crime_norm","safety_norm","investment_score"
]
display(df[display_cols].head(30))

# 11) save updated master dataset (optional)
df.to_csv("../data/master_dataset_final.csv", index=False)
print("Saved ../data/master_dataset_final.csv (with updated Crime_Index & investment_score)")


Sample df City -> City_clean


,City,City_clean
0,Chennai,Chennai
1,"Pozhichalur,Chennai",Chennai
2,"West Tambaram,Chennai",Chennai
3,"Chepauk, Triplicane,Chennai",Chennai
4,Chennai,Chennai
5,Chennai,Chennai
6,"Gowrivakkam, Sembakkam,Chennai",Chennai
7,Chennai,Chennai
8,"West Tambaram,Chennai",Chennai
9,"Chromepet,Chennai",Chennai


Sample crime_small City -> City_clean -> Crime_Index


,City,City_clean,Crime_Index
0,Ahmedabad (Gujarat),Ahmedabad,34952
1,Bengaluru(Karnataka),Bangalore,28358
2,Chennai(Tamil Nadu),Chennai,58508
3,Coimbatore(Tamil Nadu),Coimbatore,7972
4,Delhi,Delhi,99086
5,Ghaziabad(Uttar Pradesh),Ghaziabad,4277
6,Hyderabad(Telangana),Hyderabad,8200
7,Indore(Madhya Pradesh),Indore,9768
8,Jaipur(Rajasthan),Jaipur,16344
9,Kanpur(Uttar Pradesh),Kanpur,8425


crime_city sample:


,City_clean,Crime_Index
0,Ahmedabad,34952.0
1,Bangalore,28358.0
2,Chennai,58508.0
3,Coimbatore,7972.0
4,Delhi,99086.0
5,Ghaziabad,4277.0
6,Hyderabad,8200.0
7,Indore,9768.0
8,Jaipur,16344.0
9,Kanpur,8425.0


After merge - Crime_Index count / stats:
Valid (non-NA) count: 14528


count    14528.000000
mean     29736.732310
std      14867.665713
min       8200.000000
25%      17123.000000
50%      28358.000000
75%      28358.000000
max      58508.000000
Name: Crime_Index, dtype: float64

,Name,City,City_clean,Locality,Price,Avg_Rent,rental_yield_pct,Crime_Index,ry_norm,crime_norm,safety_norm,investment_score
0,Casagrand ECR 14,Chennai,Chennai,Kanathur Reddikuppam,19900000.0,14000.000000,0.844221,58508.0,8.962007e-09,1.0,0.0,0.0
1,"Ramanathan Nagar, Pozhichalur,Chennai","Pozhichalur,Chennai",Chennai,Ramanathan Nagar,22500000.0,NaN,0.000000,58508.0,0.000000e+00,1.0,0.0,0.0
2,DAC Prapthi,"West Tambaram,Chennai",Chennai,Kasthuribai Nagar,10000000.0,NaN,0.000000,58508.0,0.000000e+00,1.0,0.0,0.0
3,"Naveenilaya,Chepauk, Triplicane,Chennai","Chepauk, Triplicane,Chennai",Chennai,Naveenilaya,33300000.0,NaN,0.000000,58508.0,0.000000e+00,1.0,0.0,0.0
4,VGN Spring Field Phase 1,Chennai,Chennai,Avadi,4800000.0,8000.000000,2.000000,58508.0,2.123142e-08,1.0,0.0,0.0
5,KG Earth Homes,Chennai,Chennai,Siruseri,4000000.0,18833.333333,5.650000,58508.0,5.997877e-08,1.0,0.0,0.0
6,"THIRAN FLATS ,Gowrivakkam, Sembakkam,Chennai","Gowrivakkam, Sembakkam,Chennai",Chennai,THIRAN FLATS,6000000.0,NaN,0.000000,58508.0,0.000000e+00,1.0,0.0,0.0
7,TK Jasmine Grove,Chennai,Chennai,Mahindra World City,7235000.0,13333.333333,2.211472,58508.0,2.347635e-08,1.0,0.0,0.0
8,Avenue,"West Tambaram,Chennai",Chennai,Brindavan Colony,4200000.0,NaN,0.000000,58508.0,0.000000e+00,1.0,0.0,0.0
9,Guru Kothai Aparts,"Chromepet,Chennai",Chennai,New Colony,3000000.0,NaN,0.000000,58508.0,0.000000e+00,1.0,0.0,0.0


Saved ../data/master_dataset_final.csv (with updated Crime_Index & investment_score)


In [182]:
# diagnostics
print("rental_yield_pct stats:")
print(df["rental_yield_pct"].describe())

print("\nCrime_Index stats:")
print(df["Crime_Index"].describe())

# show largest rental yields and largest crime cities
display(df[["Name","City_clean","Locality","rental_yield_pct","Price","Avg_Rent"]]
        .sort_values("rental_yield_pct", ascending=False).head(10))

# show city crime ranks
display(crime_city.sort_values("Crime_Index", ascending=False).head(10))


rental_yield_pct stats:
count    1.452800e+04
mean     6.484229e+03
std      7.815342e+05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      9.420000e+07
Name: rental_yield_pct, dtype: float64

Crime_Index stats:
count    14528.000000
mean     29736.732310
std      14867.665713
min       8200.000000
25%      17123.000000
50%      28358.000000
75%      28358.000000
max      58508.000000
Name: Crime_Index, dtype: float64


,Name,City_clean,Locality,rental_yield_pct,Price,Avg_Rent
7948,Unique Tower by Unique Construction Mumbai,Mumbai,Mumbai Central,9.420000e+07,1.0,78500.000000
7938,"Khar West, Mumbai",Mumbai,Khar West,1.533333e+02,2100000.0,268333.333333
7979,Khar friend's,Mumbai,Khar West,7.155556e+01,4500000.0,268333.333333
4272,"Devanahalli, Bangalore",Bangalore,Devanahalli,2.258824e+01,850000.0,16000.000000
5878,Om Sai Emerald,Bangalore,Wilson Garden,2.000000e+01,720000.0,12000.000000
7847,"Kanjurmarg West, Mumbai",Mumbai,Kanjurmarg West,1.800000e+01,3000000.0,45000.000000
819,TNHB LIG Flats,Chennai,Sholinganallur,1.736842e+01,1900000.0,27500.000000
5216,Ganesha Residency by Ganesha Builder,Bangalore,Kothanur,1.711111e+01,2700000.0,38500.000000
846,"Sholinganallur, Chennai",Chennai,Sholinganallur,1.650000e+01,2000000.0,27500.000000
8971,"Barasat, Kolkata",Kolkata,Barasat,1.488000e+01,500000.0,6200.000000


,City_clean,Crime_Index
19,TOTAL CITIES,434769.0
4,Delhi,99086.0
2,Chennai,58508.0
14,Mumbai,54543.0
0,Ahmedabad,34952.0
1,Bangalore,28358.0
18,Surat,25620.0
17,Pune,17123.0
8,Jaipur,16344.0
10,Kochi,15209.0


In [183]:
import numpy as np
import pandas as pd

# 1) Make sure rental_yield_pct is numeric and fill NaN with 0
df["rental_yield_pct"] = pd.to_numeric(df["rental_yield_pct"], errors="coerce").fillna(0)

# 2) Cap rental_yield_pct at the 99th percentile to avoid huge outliers
cap_99 = df["rental_yield_pct"].quantile(0.99)
df["ry_capped"] = np.where(df["rental_yield_pct"] > cap_99,
                           cap_99,
                           df["rental_yield_pct"])

print("99th percentile cap for rental_yield_pct:", cap_99)

# 3) Percentile rank for rental yield (0..1, higher = better)
df["ry_prank"] = df["ry_capped"].rank(method="average", pct=True)

# 4) Percentile rank for crime (0..1, higher = MORE crime)
df["crime_prank"] = df["Crime_Index"].rank(method="average", pct=True)

# Convert to safety percentile (0..1, higher = safer)
df["safety_prank"] = 1.0 - df["crime_prank"]

# 5) Combine with weights (70% rental yield, 30% safety)
w_ry = 0.7
w_safety = 0.3

df["investment_score"] = (w_ry * df["ry_prank"] + w_safety * df["safety_prank"]) * 100
df["investment_score"] = df["investment_score"].round(1)

# 6) Quick sanity check – show top 20 by score
cols_show = [
    "Name","City_clean","Locality",
    "Price","Avg_Rent","rental_yield_pct","ry_capped","ry_prank",
    "Crime_Index","crime_prank","safety_prank","investment_score"
]
display(df[cols_show].sort_values("investment_score", ascending=False).head(20))


99th percentile cap for rental_yield_pct: 4.5


,Name,City_clean,Locality,Price,Avg_Rent,rental_yield_pct,ry_capped,ry_prank,Crime_Index,crime_prank,safety_prank,investment_score
6443,Town Centre Apartment,Hyderabad,Mokila,5300000.0,22000.000000,4.981132,4.500000,0.994941,8200.0,0.018619,0.981381,99.1
6434,"Gachibowli, Hyderabad",Hyderabad,Gachibowli,8000000.0,43879.310345,6.581897,4.500000,0.994941,8200.0,0.018619,0.981381,99.1
6173,Town Centre Apartment,Hyderabad,Mokila,5500000.0,22000.000000,4.800000,4.500000,0.994941,8200.0,0.018619,0.981381,99.1
6169,Shree Anurag Sri Sai Anurag New Town Phase 2,Hyderabad,Thumukunta,4000000.0,15000.000000,4.500000,4.500000,0.994941,8200.0,0.018619,0.981381,99.1
6500,"Gachibowli, Hyderabad",Hyderabad,Gachibowli,9800000.0,43879.310345,5.372977,4.500000,0.994941,8200.0,0.018619,0.981381,99.1
6438,"Gachibowli, Hyderabad",Hyderabad,Gachibowli,14000000.0,43879.310345,3.761084,3.761084,0.984650,8200.0,0.018619,0.981381,98.4
6457,Theme Golf View,Hyderabad,Nanakaramguda,14000000.0,43500.000000,3.728571,3.728571,0.984169,8200.0,0.018619,0.981381,98.3
6620,Mandeep Towers,Hyderabad,Bachupally,4700000.0,12666.666667,3.234043,3.234043,0.980727,8200.0,0.018619,0.981381,98.1
6469,Giridhari Executive Park,Hyderabad,Peeramcheru,8000000.0,20666.666667,3.100000,3.100000,0.978868,8200.0,0.018619,0.981381,98.0
6432,Ramky Ramky One Kosmos,Hyderabad,Gopanapalli,10000000.0,25666.666667,3.080000,3.080000,0.978593,8200.0,0.018619,0.981381,97.9


In [184]:
top20 = df.sort_values("investment_score", ascending=False).head(20)

cols = [
    "Name","City_clean","Locality","Price","Avg_Rent",
    "rental_yield_pct","Crime_Index","investment_score"
]

display(top20[cols])

top20[cols].to_csv("../data/top20_properties.csv", index=False)
print("✔ Saved top20_properties.csv")


,Name,City_clean,Locality,Price,Avg_Rent,rental_yield_pct,Crime_Index,investment_score
6443,Town Centre Apartment,Hyderabad,Mokila,5300000.0,22000.000000,4.981132,8200.0,99.1
6434,"Gachibowli, Hyderabad",Hyderabad,Gachibowli,8000000.0,43879.310345,6.581897,8200.0,99.1
6173,Town Centre Apartment,Hyderabad,Mokila,5500000.0,22000.000000,4.800000,8200.0,99.1
6169,Shree Anurag Sri Sai Anurag New Town Phase 2,Hyderabad,Thumukunta,4000000.0,15000.000000,4.500000,8200.0,99.1
6500,"Gachibowli, Hyderabad",Hyderabad,Gachibowli,9800000.0,43879.310345,5.372977,8200.0,99.1
6438,"Gachibowli, Hyderabad",Hyderabad,Gachibowli,14000000.0,43879.310345,3.761084,8200.0,98.4
6457,Theme Golf View,Hyderabad,Nanakaramguda,14000000.0,43500.000000,3.728571,8200.0,98.3
6620,Mandeep Towers,Hyderabad,Bachupally,4700000.0,12666.666667,3.234043,8200.0,98.1
6469,Giridhari Executive Park,Hyderabad,Peeramcheru,8000000.0,20666.666667,3.100000,8200.0,98.0
6432,Ramky Ramky One Kosmos,Hyderabad,Gopanapalli,10000000.0,25666.666667,3.080000,8200.0,97.9


✔ Saved top20_properties.csv


In [185]:
city_rank = (
    df.groupby("City_clean", as_index=False)["investment_score"]
      .mean()
      .sort_values("investment_score", ascending=False)
)

display(city_rank)

city_rank.to_csv("../data/city_ranking.csv", index=False)
print("✔ Saved city_ranking.csv")


,City_clean,investment_score
2,Hyderabad,66.050185
3,Kolkata,62.987356
6,Pune,55.700000
0,Bangalore,49.476424
5,New Delhi,45.800000
7,Thane,45.800000
4,Mumbai,39.511308
1,Chennai,38.904890


✔ Saved city_ranking.csv


In [186]:
locality_rank = (
    df.groupby(["City_clean","Locality"], as_index=False)["investment_score"]
      .mean()
      .sort_values("investment_score", ascending=False)
)

display(locality_rank.head(20))

locality_rank.to_csv("../data/locality_ranking.csv", index=False)
print("✔ Saved locality_ranking.csv")


,City_clean,Locality,investment_score
2599,Hyderabad,Mokila,99.100000
2494,Hyderabad,Gachibowli,98.866667
2505,Hyderabad,Gopanapalli,97.900000
2605,Hyderabad,Nanakaramguda,97.750000
2726,Hyderabad,Thumukunta,97.700000
2585,Hyderabad,Manneguda,97.500000
2626,Hyderabad,Peeramcheru,97.250000
2614,Hyderabad,Nizampet,97.200000
2459,Hyderabad,Bandlaguda Jagir,97.200000
2643,Hyderabad,Puppalaguda,97.150000


✔ Saved locality_ranking.csv


In [187]:
weather.columns
weather.head()


,event_date,state,event_type,duration_days,precipitation_anomaly_mm,mei_index,temperature_anomaly_c,wind_anomaly_kmph,damage_estimate_millionusd,intensity_scale
0,21-01-2016,Odisha,Flood,6,281.61,-2.46,3.09,56.25,426.99,5
1,01-10-2010,Telangana,Cloudburst,21,230.75,0.09,-4.43,35.20,454.11,4
2,16-02-2021,Madhya Pradesh,Hailstorm,29,461.28,0.88,5.51,110.85,447.89,10
3,02-01-2023,Delhi,Landslide,5,294.12,-0.67,0.26,99.52,384.12,9
4,03-01-2016,Tripura,Landslide,10,453.21,0.72,1.91,76.62,21.11,4


In [188]:
import pandas as pd

# 1) Clean column names
weather.columns = weather.columns.str.strip().str.lower()

# 2) Filter only flood-related events
flood_events = weather[weather["event_type"].str.contains("flood", case=False, na=False)].copy()

# 3) Create a simple flood risk metric
flood_events.loc[:, "flood_risk_raw"] = (
    flood_events["damage_estimate_millionusd"] / flood_events["intensity_scale"]
)

# 4) Average flood risk per state
flood_state = (
    flood_events.groupby("state", as_index=False)["flood_risk_raw"].mean()
)

flood_state.rename(columns={
    "state": "State",
    "flood_risk_raw": "Flood_Risk"
}, inplace=True)

print(flood_state.head())


               State  Flood_Risk
0     Andhra Pradesh   72.055450
1  Arunachal Pradesh   78.464942
2              Assam   76.589758
3              Bihar   65.103161
4       Chhattisgarh   80.497541


In [189]:
city_to_state = {
    "Mumbai": "Maharashtra",
    "Thane": "Maharashtra",
    "Pune": "Maharashtra",

    "Bangalore": "Karnataka",
    "Bengaluru": "Karnataka",

    "Chennai": "Tamil Nadu",

    "Hyderabad": "Telangana",

    "Delhi": "Delhi",
    "New Delhi": "Delhi",

    "Kolkata": "West Bengal",
    "Jaipur": "Rajasthan",
    "Ahmedabad": "Gujarat",
    "Surat": "Gujarat",
    "Nagpur": "Maharashtra",
    "Nashik": "Maharashtra",
    "Lucknow": "Uttar Pradesh",
    # add more based on your City values if needed
}


In [190]:
df["City"].head(20).apply(lambda x: (x, type(x)))


0                              (Chennai, <class 'str'>)
1                  (Pozhichalur,Chennai, <class 'str'>)
2                (West Tambaram,Chennai, <class 'str'>)
3          (Chepauk, Triplicane,Chennai, <class 'str'>)
4                              (Chennai, <class 'str'>)
5                              (Chennai, <class 'str'>)
6       (Gowrivakkam, Sembakkam,Chennai, <class 'str'>)
7                              (Chennai, <class 'str'>)
8                (West Tambaram,Chennai, <class 'str'>)
9                    (Chromepet,Chennai, <class 'str'>)
10                             (Chennai, <class 'str'>)
11                (Pallikaranai,Chennai, <class 'str'>)
12               (West Mambalam,Chennai, <class 'str'>)
13    (Mettuppalayam, Ashok Nagar,Chennai, <class 's...
14                (Madambakkam, Chennai, <class 'str'>)
15                             (Chennai, <class 'str'>)
16                   (Padappai, Chennai, <class 'str'>)
17                (Nungambakkam,Chennai, <class 

In [191]:
df["City_clean"] = (
    df["City"]
    .astype(str)
    .str.replace(r"[\(\)\[\]]", "", regex=True)  # remove () and []
    .str.replace(r"'", "", regex=True)           # remove single quotes
    .str.replace(r"  ", " ")                     # remove double spaces
    .str.strip()
)


In [192]:
df["Main_City"] = (
    df["City_clean"]
    .str.split(",", expand=True)
    .iloc[:, -1]
    .str.strip()
)


In [193]:
# Extract last city part (not first or middle)

# Let’s replace your earlier logic with this one:
df["Main_City"] = (
    df["City_clean"]
    .str.split(",")          # split on ALL commas
    .apply(lambda parts: parts[-1].strip())   # take LAST part
)


In [194]:
df[["City_clean", "Main_City"]].head(20)


,City_clean,Main_City
0,Chennai,Chennai
1,"Pozhichalur,Chennai",Chennai
2,"West Tambaram,Chennai",Chennai
3,"Chepauk, Triplicane,Chennai",Chennai
4,Chennai,Chennai
5,Chennai,Chennai
6,"Gowrivakkam, Sembakkam,Chennai",Chennai
7,Chennai,Chennai
8,"West Tambaram,Chennai",Chennai
9,"Chromepet,Chennai",Chennai


In [195]:
df["State"] = df["Main_City"].map(city_to_state)
df[["Main_City","State"]].head(20)


,Main_City,State
0,Chennai,Tamil Nadu
1,Chennai,Tamil Nadu
2,Chennai,Tamil Nadu
3,Chennai,Tamil Nadu
4,Chennai,Tamil Nadu
5,Chennai,Tamil Nadu
6,Chennai,Tamil Nadu
7,Chennai,Tamil Nadu
8,Chennai,Tamil Nadu
9,Chennai,Tamil Nadu


In [196]:
# See which flood columns exist
[c for c in df.columns if "flood" in c.lower()]


[]

In [197]:
# Drop old Flood_Risk columns
cols_to_drop = [c for c in df.columns if "flood" in c.lower()]
print("Dropping:", cols_to_drop)

df = df.drop(columns=cols_to_drop, errors="ignore")


Dropping: []


In [198]:
print(flood_state.columns)   # should be ['State', 'Flood_Risk']


Index(['State', 'Flood_Risk'], dtype='object')


In [199]:
# Merge cleanly with flood_state
df = df.merge(flood_state, on="State", how="left")

df[["City_clean","Main_City","State","Flood_Risk"]].head(20)


,City_clean,Main_City,State,Flood_Risk
0,Chennai,Chennai,Tamil Nadu,72.202177
1,"Pozhichalur,Chennai",Chennai,Tamil Nadu,72.202177
2,"West Tambaram,Chennai",Chennai,Tamil Nadu,72.202177
3,"Chepauk, Triplicane,Chennai",Chennai,Tamil Nadu,72.202177
4,Chennai,Chennai,Tamil Nadu,72.202177
5,Chennai,Chennai,Tamil Nadu,72.202177
6,"Gowrivakkam, Sembakkam,Chennai",Chennai,Tamil Nadu,72.202177
7,Chennai,Chennai,Tamil Nadu,72.202177
8,"West Tambaram,Chennai",Chennai,Tamil Nadu,72.202177
9,"Chromepet,Chennai",Chennai,Tamil Nadu,72.202177


In [200]:
df["Flood_Risk"] = df["Flood_Risk"].fillna(df["Flood_Risk"].median())
df["Flood_Risk"].head()

0    72.202177
1    72.202177
2    72.202177
3    72.202177
4    72.202177
Name: Flood_Risk, dtype: float64

In [201]:
df.to_csv("../data/master_dataset_final_with_scores.csv", index=False)
print("✅ Saved master_dataset_final_with_scores.csv")


✅ Saved master_dataset_final_with_scores.csv


In [202]:
# Convert to numeric in case of stray strings
import pandas as pd
df["Flood_Risk"] = pd.to_numeric(df["Flood_Risk"], errors="coerce")

# Normalize so LOWER flood risk = HIGHER score
df["flood_norm"] = 1 - (
    (df["Flood_Risk"] - df["Flood_Risk"].min()) /
    (df["Flood_Risk"].max() - df["Flood_Risk"].min())
)

df["flood_norm"] = df["flood_norm"].clip(0,1)  # safety
df["flood_norm"].head()


0    0.059771
1    0.059771
2    0.059771
3    0.059771
4    0.059771
Name: flood_norm, dtype: float64

In [203]:
# Final investment score including flood risk
df["investment_score_final"] = (
    0.7 * df["ry_prank"] + 
    0.2 * df["safety_prank"] + 
    0.1 * df["flood_norm"]
) * 100

df["investment_score_final"] = df["investment_score_final"].round(1)

df[[
    "Name","City_clean","Locality","rental_yield_pct",
    "Crime_Index","Flood_Risk","ry_prank","safety_prank","flood_norm",
    "investment_score","investment_score_final"
]].head(20)


,Name,City_clean,Locality,rental_yield_pct,Crime_Index,Flood_Risk,ry_prank,safety_prank,flood_norm,investment_score,investment_score_final
0,Casagrand ECR 14,Chennai,Kanathur Reddikuppam,0.844221,58508.0,72.202177,0.944039,0.05486,0.059771,67.7,67.8
1,"Ramanathan Nagar, Pozhichalur,Chennai","Pozhichalur,Chennai",Ramanathan Nagar,0.000000,58508.0,72.202177,0.468096,0.05486,0.059771,34.4,34.5
2,DAC Prapthi,"West Tambaram,Chennai",Kasthuribai Nagar,0.000000,58508.0,72.202177,0.468096,0.05486,0.059771,34.4,34.5
3,"Naveenilaya,Chepauk, Triplicane,Chennai","Chepauk, Triplicane,Chennai",Naveenilaya,0.000000,58508.0,72.202177,0.468096,0.05486,0.059771,34.4,34.5
4,VGN Spring Field Phase 1,Chennai,Avadi,2.000000,58508.0,72.202177,0.964723,0.05486,0.059771,69.2,69.2
5,KG Earth Homes,Chennai,Siruseri,5.650000,58508.0,72.202177,0.994941,0.05486,0.059771,71.3,71.3
6,"THIRAN FLATS ,Gowrivakkam, Sembakkam,Chennai","Gowrivakkam, Sembakkam,Chennai",THIRAN FLATS,0.000000,58508.0,72.202177,0.468096,0.05486,0.059771,34.4,34.5
7,TK Jasmine Grove,Chennai,Mahindra World City,2.211472,58508.0,72.202177,0.968750,0.05486,0.059771,69.5,69.5
8,Avenue,"West Tambaram,Chennai",Brindavan Colony,0.000000,58508.0,72.202177,0.468096,0.05486,0.059771,34.4,34.5
9,Guru Kothai Aparts,"Chromepet,Chennai",New Colony,0.000000,58508.0,72.202177,0.468096,0.05486,0.059771,34.4,34.5


In [204]:
# Resilient merge + rental yield fix (run this entire cell)
import pandas as pd
import numpy as np

# 0) quick check: show available columns in df and rent_grouped
print("df columns:", df.columns.tolist())
print("rent_grouped columns:", rent_grouped.columns.tolist())

# 1) Ensure we have safe string city/locality fields to work with
df["City"] = df["City"].astype(str)
df["Locality"] = df["Locality"].astype(str)

# 2) Create City_clean = last chunk after comma (main city)
df["City_clean"] = df["City"].str.split(",").apply(lambda parts: parts[-1].strip() if len(parts) else "")

# 3) Create Locality_clean by removing a trailing city from Locality if present:
def clean_locality(val):
    parts = [p.strip() for p in str(val).split(",") if p is not None and str(p).strip() != ""]
    if len(parts) <= 1:
        return parts[0] if parts else ""
    # If last part equals the city_clean, drop it; otherwise join all but last (safer)
    if parts[-1].lower() == df.loc[val_index := 0, "City_clean"].lower() if False else None:
        pass
    # General rule: remove final token if it looks like a city (contains same text as City_clean)
    # We'll do a generic drop-last so "Chepauk, Triplicane, Chennai" -> "Chepauk, Triplicane"
    return ", ".join(parts[:-1]).strip()

# Simpler robust locality cleaning (safer)
df["Locality_clean"] = df["Locality"].apply(lambda v: ", ".join([p.strip() for p in str(v).split(",")[:-1]]).strip() 
                                           if ("," in str(v)) else str(v).strip())
# If the above leaves empty strings (because locality had single token), fallback to original
df["Locality_clean"] = df["Locality_clean"].replace("", np.nan).fillna(df["Locality"].str.strip())

# 4) Show sample to confirm cleaning
print("\nSample cleaned City / Locality")
display(df[["City","City_clean","Locality","Locality_clean"]].head(12))

# 5) Detect whether Avg_Rent already exists, or any rent-like column exists
if "Avg_Rent" in df.columns:
    print("\nAvg_Rent already present in df (no merge needed).")
else:
    # look for rent-like columns present in df after earlier merges
    rent_like = [c for c in df.columns if "rent" in c.lower()]
    print("\nNo Avg_Rent column found in df. rent-like columns found in df:", rent_like)
    # check rent_grouped existence & columns
    if "rent_grouped" in globals():
        print("rent_grouped exists with columns:", rent_grouped.columns.tolist())
    else:
        print("rent_grouped not found in current namespace. Make sure rent_grouped is defined.")

# 6) Perform merge using City_clean & Locality_clean -> rent_grouped["City","Locality"]
# Only merge if Avg_Rent not present or merge forced
if "Avg_Rent" not in df.columns:
    # Make sure rent_grouped has City & Locality names expected
    rg = rent_grouped.copy()
    rg["City"] = rg["City"].astype(str).str.strip()
    rg["Locality"] = rg["Locality"].astype(str).str.strip()
    # Merge (left join to preserve df rows)
    df = df.merge(rg[["City","Locality","Avg_Rent"]], 
                  left_on=["City_clean","Locality_clean"],
                  right_on=["City","Locality"],
                  how="left",
                  suffixes=("","_rent"))
    # If merge introduced duplicate Avg_Rent columns with different names, pick the correct one:
    # Prefer newly merged 'Avg_Rent' in df; if not present, try columns that contain 'Avg_Rent' substring
    if "Avg_Rent" not in df.columns:
        possible = [c for c in df.columns if "avg" in c.lower() and "rent" in c.lower()]
        if possible:
            df["Avg_Rent"] = df[possible[0]]
            print(f"Using rent column '{possible[0]}' as Avg_Rent")
        else:
            print("Warning: After merge no Avg_Rent column found. rent_grouped might have different naming.")

# 7) Ensure Avg_Rent exists now; if not, create as 0
if "Avg_Rent" not in df.columns:
    df["Avg_Rent"] = 0.0
    print("Created Avg_Rent with zeros (fallback).")

# 8) Fill NaN Avg_Rent with 0 (or you can keep NaN if you prefer)
df["Avg_Rent"] = pd.to_numeric(df["Avg_Rent"], errors="coerce").fillna(0)

# 9) Compute rental_yield_pct safely (avoid division by zero)
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["rental_yield_pct"] = 0.0
mask = (df["Price"] > 0)
df.loc[mask, "rental_yield_pct"] = (df.loc[mask, "Avg_Rent"] * 12) / df.loc[mask, "Price"] * 100

# 10) diagnostics: how many Avg_Rent zeros and how many rental_yield_pct zeros
print("\nDiagnostics:")
print("Rows total:", len(df))
print("Avg_Rent == 0 count:", int((df["Avg_Rent"]==0).sum()))
print("rental_yield_pct == 0 count:", int((df["rental_yield_pct"]==0).sum()))
print("Sample rows with nonzero Avg_Rent / rental_yield_pct:")
display(df[df["rental_yield_pct"]>0][["Name","City","City_clean","Locality","Locality_clean","Avg_Rent","Price","rental_yield_pct"]].head(20))

# 11) If many zeros remain, show an example where City_clean matches rent_grouped City but locality mismatch exists
example_city = df["City_clean"].value_counts().idxmax()
print(f"\nExample city used for diagnostic: {example_city}")
display(rent_grouped[rent_grouped["City"].str.contains(example_city, na=False)].head(10))

# Save df back to workspace
print("\nDone. If rental_yield_pct is still 0 for many rows, we will need to improve locality matching (e.g. abbreviations, alternate spellings).")


df columns: ['Name', 'Property Title', 'Price', 'Area_sqft', 'Locality', 'City', 'Avg_Rent', 'rental_yield_pct', 'ry_norm', 'crime_norm', 'safety_norm', 'investment_score', 'City_clean', 'Crime_Index', 'ry_capped', 'ry_prank', 'crime_prank', 'safety_prank', 'Main_City', 'State', 'Flood_Risk', 'flood_norm', 'investment_score_final']
rent_grouped columns: ['City', 'Locality', 'Avg_Rent']

Sample cleaned City / Locality


,City,City_clean,Locality,Locality_clean
0,Chennai,Chennai,Kanathur Reddikuppam,Kanathur Reddikuppam
1,"Pozhichalur,Chennai",Chennai,Ramanathan Nagar,Ramanathan Nagar
2,"West Tambaram,Chennai",Chennai,Kasthuribai Nagar,Kasthuribai Nagar
3,"Chepauk, Triplicane,Chennai",Chennai,Naveenilaya,Naveenilaya
4,Chennai,Chennai,Avadi,Avadi
5,Chennai,Chennai,Siruseri,Siruseri
6,"Gowrivakkam, Sembakkam,Chennai",Chennai,THIRAN FLATS,THIRAN FLATS
7,Chennai,Chennai,Mahindra World City,Mahindra World City
8,"West Tambaram,Chennai",Chennai,Brindavan Colony,Brindavan Colony
9,"Chromepet,Chennai",Chennai,New Colony,New Colony



Avg_Rent already present in df (no merge needed).

Diagnostics:
Rows total: 14528
Avg_Rent == 0 count: 13600
rental_yield_pct == 0 count: 13600
Sample rows with nonzero Avg_Rent / rental_yield_pct:


,Name,City,City_clean,Locality,Locality_clean,Avg_Rent,Price,rental_yield_pct
0,Casagrand ECR 14,Chennai,Chennai,Kanathur Reddikuppam,Kanathur Reddikuppam,14000.000000,19900000.0,0.844221
4,VGN Spring Field Phase 1,Chennai,Chennai,Avadi,Avadi,8000.000000,4800000.0,2.000000
5,KG Earth Homes,Chennai,Chennai,Siruseri,Siruseri,18833.333333,4000000.0,5.650000
7,TK Jasmine Grove,Chennai,Chennai,Mahindra World City,Mahindra World City,13333.333333,7235000.0,2.211472
10,Mahindra Lifespaces Aqualilly Flexi Homes,Chennai,Chennai,Mahindra World City,Mahindra World City,13333.333333,2940000.0,5.442177
30,Redmond Square,Chennai,Chennai,Sholinganallur,Sholinganallur,27500.000000,7000000.0,4.714286
35,Jones Dawn Villas by Jones Foundations,Chennai,Chennai,Ponmar,Ponmar,10500.000000,8700000.0,1.448276
48,Krishna Apartment,Chennai,Chennai,Madambakkam,Madambakkam,11875.000000,5600000.0,2.544643
50,"Madambakkam, Chennai",Chennai,Chennai,Madambakkam,Madambakkam,11875.000000,4500000.0,3.166667
62,"Velachery, Chennai",Chennai,Chennai,Velachery,Velachery,16772.727273,7900000.0,2.547756



Example city used for diagnostic: Bangalore


,City,Locality,Avg_Rent
0,Bangalore,"A Narayanapura, Mahadevapura",14500.000000
1,Bangalore,Aarna Enclave,90000.000000
2,Bangalore,"Abbiareddy Layout, Kaggadasapura",22500.000000
3,Bangalore,Abbigere,9333.333333
4,Bangalore,"Aditya Nagar-Vidyaranyapura, Vidyaranyapura",10000.000000
5,Bangalore,Adugodi,10500.000000
6,Bangalore,Aduru,6500.000000
7,Bangalore,"Aecs Layout-Singasandra, Singasandra, Hosur Road",8400.000000
8,Bangalore,Agrahara Layout,12000.000000
9,Bangalore,"Ags Layout, Hebbal",20000.000000



Done. If rental_yield_pct is still 0 for many rows, we will need to improve locality matching (e.g. abbreviations, alternate spellings).


In [205]:
# 1a) Make sure flood_norm exists and is numeric
import pandas as pd
import numpy as np

# convert to numeric if needed
df["Flood_Risk"] = pd.to_numeric(df.get("Flood_Risk", pd.Series(np.nan)), errors="coerce")
df["flood_norm"] = pd.to_numeric(df.get("flood_norm", pd.Series(np.nan)), errors="coerce")

# if flood_norm missing, fill with median (so it doesn't break)
if df["flood_norm"].isna().all():
    df["flood_norm"] = 0.5   # neutral if no flood info
else:
    df["flood_norm"] = df["flood_norm"].fillna(df["flood_norm"].median())

# 1b) Ensure crime-based safety_prank exists (1 - crime_prank)
# If crime_prank missing, compute it:
if "crime_prank" not in df.columns:
    df["crime_prank"] = df["Crime_Index"].rank(method="average", pct=True)
df["safety_prank"] = 1.0 - df["crime_prank"]

# 1c) Combine them into a single safety measure.
# Example weights: 60% crime safety, 40% flood safety (tune as needed)
w_crime = 0.6
w_flood = 0.4
df["safety_final"] = (w_crime * df["safety_prank"] + w_flood * df["flood_norm"])
# keep in 0..1
df["safety_final"] = df["safety_final"].clip(0,1)


In [206]:
# 2a) Ensure ry_prank exists (percentile rank of rental yield)
if "ry_prank" not in df.columns:
    # cap extreme yields first (optional)
    cap = df["rental_yield_pct"].quantile(0.99)
    df["ry_capped"] = np.where(df["rental_yield_pct"] > cap, cap, df["rental_yield_pct"])
    df["ry_prank"] = df["ry_capped"].rank(method="average", pct=True)

# 2b) reweight and compute investment_score_final
w_ry = 0.7   # weight for rental yield (you used 0.7 earlier)
w_safety = 0.3

# If you want flood to matter more, reduce w_ry slightly and depend more on safety_final.
# Example: w_ry=0.6, w_safety=0.4  (change above)

df["investment_score_final"] = (w_ry * df["ry_prank"] + w_safety * df["safety_final"]) * 100
df["investment_score_final"] = df["investment_score_final"].round(1).clip(0,100)

# quick check
df[["Name","City_clean","Locality","rental_yield_pct","ry_prank","safety_final","investment_score_final"]].sort_values(
    "investment_score_final", ascending=False).head(20)


,Name,City_clean,Locality,rental_yield_pct,ry_prank,safety_final,investment_score_final
8864,West WBHB Belaghata,Kolkata,Beliaghata,5.833016,0.994941,0.847911,95.1
9146,"Barrackpore, Kolkata",Kolkata,Barrackpore,6.352941,0.994941,0.847911,95.1
8067,"Barrackpore, Kolkata",Kolkata,Barrackpore,9.000000,0.994941,0.847911,95.1
9100,Regent Estate,Kolkata,Bijoygarh,4.752000,0.994941,0.847911,95.1
8971,"Barasat, Kolkata",Kolkata,Barasat,14.880000,0.994941,0.847911,95.1
8177,"Bijoygarh, Kolkata",Kolkata,Bijoygarh,5.940000,0.994941,0.847911,95.1
8859,"Beliaghata, Kolkata",Kolkata,Beliaghata,5.100000,0.994941,0.847911,95.1
8957,"Parnasree Pally, Kolkata",Kolkata,Parnasree Pally,4.888889,0.994941,0.847911,95.1
8851,UD 16 Parnashree Palli,Kolkata,Parnasree Pally,6.600000,0.994941,0.847911,95.1
9104,Sanhita Township Project,Kolkata,New Town,4.571429,0.994941,0.847911,95.1


In [207]:
# A) treat 0 as missing and replace with city median
df["Avg_Rent"] = pd.to_numeric(df.get("Avg_Rent", pd.Series(np.nan)), errors="coerce")
df["Avg_Rent"].replace(0, np.nan, inplace=True)

# compute city median from rent_grouped (if you have it), else from df
if 'rent_grouped' in globals():
    # ensure consistent names
    rent_grouped["City"] = rent_grouped["City"].astype(str).str.strip()
    rent_grouped["Locality"] = rent_grouped["Locality"].astype(str).str.strip()
    # city median from rent_grouped
    city_median = rent_grouped.groupby("City")["Avg_Rent"].median().to_dict()
    df["Avg_Rent_from_city"] = df["City_clean"].map(city_median)
    # fill Avg_Rent with city median where missing
    df["Avg_Rent"] = df["Avg_Rent"].fillna(df["Avg_Rent_from_city"])
else:
    # compute from df itself
    df["Avg_Rent"] = df["Avg_Rent"].fillna(df.groupby("City_clean")["Avg_Rent"].transform("median"))

# B) fallback: fill remaining with overall median
overall_median = df["Avg_Rent"].median()
df["Avg_Rent"] = df["Avg_Rent"].fillna(overall_median)

# C) recompute rental_yield_pct safely
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
mask = df["Price"] > 0
df["rental_yield_pct"] = 0.0
df.loc[mask, "rental_yield_pct"] = (df.loc[mask, "Avg_Rent"] * 12) / df.loc[mask, "Price"] * 100
df["rental_yield_pct"] = df["rental_yield_pct"].replace([np.inf, -np.inf], 0).fillna(0)


C:\Users\suraj\AppData\Local\Temp\ipykernel_17864\3438106029.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Avg_Rent"].replace(0, np.nan, inplace=True)


In [208]:
import numpy as np
import pandas as pd

# ensure Avg_Rent exists and is numeric
df["Avg_Rent"] = pd.to_numeric(df.get("Avg_Rent", pd.Series(np.nan, index=df.index)), errors="coerce")

# replace zeros with NaN (do not use inplace on a column view)
df["Avg_Rent"] = df["Avg_Rent"].replace(0, np.nan)

# compute city median from rent_grouped (if available)
if "rent_grouped" in globals():
    rg = rent_grouped.copy()   # operate on a copy to avoid modifying original
    rg["City"] = rg["City"].astype(str).str.strip()
    rg["Locality"] = rg["Locality"].astype(str).str.strip()
    city_median = rg.groupby("City")["Avg_Rent"].median().to_dict()
    # map medians into df; this creates a new column we can use to fill missing values
    df["Avg_Rent_from_city"] = df["City_clean"].map(city_median)
    # fill missing Avg_Rent with city median where available
    df["Avg_Rent"] = df["Avg_Rent"].fillna(df["Avg_Rent_from_city"])
else:
    # fallback: fill from df (group transform) — reassign to avoid chained assignment
    df["Avg_Rent"] = df["Avg_Rent"].fillna(df.groupby("City_clean")["Avg_Rent"].transform("median"))

# B) fallback: fill any remaining missing values with overall median
overall_median = float(df["Avg_Rent"].median(skipna=True))
df["Avg_Rent"] = df["Avg_Rent"].fillna(overall_median)

# C) recompute rental_yield_pct safely (avoid division by zero; mask Price>0)
df["Price"] = pd.to_numeric(df.get("Price", pd.Series(np.nan, index=df.index)), errors="coerce")
mask = df["Price"] > 0
df["rental_yield_pct"] = 0.0
df.loc[mask, "rental_yield_pct"] = (df.loc[mask, "Avg_Rent"] * 12) / df.loc[mask, "Price"] * 100

# protect against infinities / NaNs
df["rental_yield_pct"] = df["rental_yield_pct"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# quick diagnostics
print("Avg_Rent missing after fill:", int(df["Avg_Rent"].isna().sum()))
print("rental_yield_pct missing or 0 count:", int((df["rental_yield_pct"] == 0).sum()))
print("rental_yield_pct stats:")
print(df["rental_yield_pct"].describe())


Avg_Rent missing after fill: 0
rental_yield_pct missing or 0 count: 1
rental_yield_pct stats:
count    1.452800e+04
mean     7.504686e+03
std      7.862967e+05
min      0.000000e+00
25%      1.576364e+00
50%      2.763068e+00
75%      4.816667e+00
max      9.420000e+07
Name: rental_yield_pct, dtype: float64


In [209]:
# ---------------------------
# POST-PROCESSING & SCORING
# (run this immediately after your rental_yield recompute cell)
# ---------------------------

import numpy as np
import pandas as pd

# 0) Safety: ensure df exists
if "df" not in globals():
    raise RuntimeError("DataFrame 'df' not found in the workspace. Make sure you ran previous steps.")

# Helper: ensure numeric columns are numeric
df["Price"] = pd.to_numeric(df.get("Price", pd.Series(np.nan, index=df.index)), errors="coerce")
df["Avg_Rent"] = pd.to_numeric(df.get("Avg_Rent", pd.Series(np.nan, index=df.index)), errors="coerce")
df["rental_yield_pct"] = pd.to_numeric(df.get("rental_yield_pct", pd.Series(np.nan, index=df.index)), errors="coerce")

# 1) Cap extremely large rental yields (99th percentile)
cap_99 = df["rental_yield_pct"].dropna().quantile(0.99) if df["rental_yield_pct"].notna().any() else 0.0
df["ry_capped"] = np.where(df["rental_yield_pct"] > cap_99, cap_99, df["rental_yield_pct"].fillna(0.0))

print(f"Applied 99th percentile cap to rental_yield_pct: {cap_99:.6f}")

# 2) Percentile rank for rental yield (higher -> better)
df["ry_prank"] = df["ry_capped"].rank(method="average", pct=True).fillna(0.0)

# 3) Crime -> ensure numeric and fill missing with median (or 0 if none present)
df["Crime_Index"] = pd.to_numeric(df.get("Crime_Index", pd.Series(np.nan, index=df.index)), errors="coerce")
if df["Crime_Index"].notna().sum() == 0:
    df["Crime_Index"] = 0.0
else:
    # preserve existing values but replace remaining NaNs by city median if possible
    # First compute per-city median if City_clean exists
    if "City_clean" in df.columns:
        city_med = df.groupby("City_clean")["Crime_Index"].median()
        df["Crime_Index"] = df["Crime_Index"].fillna(df["City_clean"].map(city_med))
    df["Crime_Index"] = df["Crime_Index"].fillna(df["Crime_Index"].median())

# 4) Crime percentile -> convert to safety (higher = safer)
df["crime_prank"] = df["Crime_Index"].rank(method="average", pct=True).fillna(0.0)
df["safety_prank"] = 1.0 - df["crime_prank"]

# 5) Flood risk: ensure numeric, fill missing by state-level median or overall median
df["Flood_Risk"] = pd.to_numeric(df.get("Flood_Risk", pd.Series(np.nan, index=df.index)), errors="coerce")
if df["Flood_Risk"].notna().sum() == 0:
    # no flood data — create neutral values (0.5 safety)
    df["flood_norm"] = 0.5
else:
    # fill na using median
    df["Flood_Risk"] = df["Flood_Risk"].fillna(df["Flood_Risk"].median())
    fmin = df["Flood_Risk"].min()
    fmax = df["Flood_Risk"].max()
    if fmax == fmin:
        df["flood_norm"] = 0.5  # no variation
    else:
        df["flood_norm"] = 1.0 - ((df["Flood_Risk"] - fmin) / (fmax - fmin))
        df["flood_norm"] = df["flood_norm"].clip(0, 1)

# 6) Combine safety (crime + flood)
# Give equal weight to crime safety and flood safety inside safety_combined.
df["safety_combined"] = (df["safety_prank"].fillna(0.0) + df["flood_norm"].fillna(0.5)) / 2.0

# 7) Final investment score: combine rental yield (70%) + safety_combined (30%)
w_ry = 0.7
w_safety = 0.3

df["investment_score"] = (w_ry * df["ry_prank"] + w_safety * df["safety_combined"]) * 100.0
df["investment_score"] = df["investment_score"].clip(0, 100).round(1)

# 8) Extra diagnostics & derived columns (optional but useful)
# - mark properties with Price <= 0 as invalid
df["valid_price"] = np.where(df["Price"] > 0, True, False)

# 9) Show diagnostics
print("\nDiagnostics:")
print(f" Total rows: {len(df)}")
print(f" Avg_Rent missing or zero (after fill): {int((df['Avg_Rent'].isna() | (df['Avg_Rent']==0)).sum())}")
print(f" rental_yield_pct zeros (==0): {int((df['rental_yield_pct']==0).sum())}")
print(f" Number invalid price rows (Price<=0): {int((df['Price'] <= 0).sum())}")

print("\nrental_yield_pct summary:")
print(df["rental_yield_pct"].describe())

print("\nCrime_Index summary:")
print(df["Crime_Index"].describe())

print("\nFlood_Risk summary:")
print(df["Flood_Risk"].describe())

# 10) Top-N properties by investment_score
cols_show = [
    "Name", "City_clean", "Locality", "Price", "Avg_Rent",
    "rental_yield_pct", "ry_capped", "ry_prank",
    "Crime_Index", "flood_norm", "safety_combined",
    "investment_score", "valid_price"
]

print("\nTop 20 properties by investment_score:")
display(df.loc[df["valid_price"], cols_show].sort_values("investment_score", ascending=False).head(20))

# 11) Save results (optional - uncomment if you want file output)
out_path = "../data/master_dataset_scored.csv"
try:
    df.to_csv(out_path, index=False)
    print(f"\nSaved scored dataset to: {out_path}")
except Exception as e:
    print(f"\nWarning: could not save CSV to {out_path}: {e}")

# 12) (Optional) Compute a city-level summary to help verify
city_summary = df.groupby("City_clean", dropna=False).agg(
    properties=("Name", "count"),
    avg_price=("Price", "median"),
    avg_rental_yield_pct=("rental_yield_pct", "median"),
    avg_crime_index=("Crime_Index", "median"),
    avg_flood_norm=("flood_norm", "median"),
    avg_investment_score=("investment_score", "median")
).reset_index().sort_values("avg_investment_score", ascending=False)

print("\nTop 10 cities by median investment_score:")
display(city_summary.head(10))

# Done
print("\nScoring complete. If results look odd, common next checks:")
print(" - verify Locality matching with rent_grouped (alternate spellings/abbreviations).")
print(" - inspect rows with rental_yield_pct == 0 to see why Avg_Rent was not found.")
print(" - if Flood_Risk is missing/flat, consider enriching weather mapping by state.")


Applied 99th percentile cap to rental_yield_pct: 26.398524

Diagnostics:
 Total rows: 14528
 Avg_Rent missing or zero (after fill): 0
 rental_yield_pct zeros (==0): 1
 Number invalid price rows (Price<=0): 0

rental_yield_pct summary:
count    1.452800e+04
mean     7.504686e+03
std      7.862967e+05
min      0.000000e+00
25%      1.576364e+00
50%      2.763068e+00
75%      4.816667e+00
max      9.420000e+07
Name: rental_yield_pct, dtype: float64

Crime_Index summary:
count    14528.000000
mean     29736.732310
std      14867.665713
min       8200.000000
25%      17123.000000
50%      28358.000000
75%      28358.000000
max      58508.000000
Name: Crime_Index, dtype: float64

Flood_Risk summary:
count    14528.000000
mean        68.738359
std          4.369140
min         62.821361
25%         62.821361
50%         70.832899
75%         72.798526
max         72.798526
Name: Flood_Risk, dtype: float64

Top 20 properties by investment_score:


,Name,City_clean,Locality,Price,Avg_Rent,rental_yield_pct,ry_capped,ry_prank,Crime_Index,flood_norm,safety_combined,investment_score,valid_price
12211,"Lake Town,Katraj, Pune",Pune,Lake Town,650000.0,14450.0,26.676923,26.398524,0.995010,17123.0,1.0,0.882486,96.1,True
12128,"Hause,Kalewadi, Pune",Pune,Hause,250000.0,14450.0,69.360000,26.398524,0.995010,17123.0,1.0,0.882486,96.1,True
11984,"Kondhwa, Pune",Pune,Kondhwa,650000.0,14450.0,26.676923,26.398524,0.995010,17123.0,1.0,0.882486,96.1,True
11964,"Anandgram,Yavat, Pune",Pune,Anandgram,110000.0,14450.0,157.636364,26.398524,0.995010,17123.0,1.0,0.882486,96.1,True
11941,"Sai Samarth Apartment,Vijay Nagar, Kalewadi,Pune",Pune,Sai Samarth Apartment,600000.0,14450.0,28.900000,26.398524,0.995010,17123.0,1.0,0.882486,96.1,True
10801,"Siddhivinayak society Ambegaon,Jambhulwadi, Pune",Pune,Siddhivinayak society Ambegaon,160000.0,14450.0,108.375000,26.398524,0.995010,17123.0,1.0,0.882486,96.1,True
12064,"Newale Vasti, Chikhali,Pune",Pune,Newale Vasti,500000.0,14450.0,34.680000,26.398524,0.995010,17123.0,1.0,0.882486,96.1,True
11993,"Khadki, Pune",Pune,Khadki,100000.0,14450.0,173.400000,26.398524,0.995010,17123.0,1.0,0.882486,96.1,True
11166,Xrbia Hinjewadi Road Riverfront Ph 1,Pune,Bebadohal,100000.0,14450.0,173.400000,26.398524,0.995010,17123.0,1.0,0.882486,96.1,True
11836,Maple Aapla Ghar Lonikand,Pune,Lonikand,300000.0,14450.0,57.800000,26.398524,0.995010,17123.0,1.0,0.882486,96.1,True



Saved scored dataset to: ../data/master_dataset_scored.csv

Top 10 cities by median investment_score:


,City_clean,properties,avg_price,avg_rental_yield_pct,avg_crime_index,avg_flood_norm,avg_investment_score
4,Mumbai,1353,9000000.0,6.315789,54543.0,1.000000,76.10
6,Pune,2964,4300000.0,4.032558,17123.0,1.000000,74.30
7,Thane,6,5650000.0,3.071191,28358.0,1.000000,60.55
3,Kolkata,1392,4303500.0,2.372093,14492.0,0.747444,54.50
5,New Delhi,2165,6250000.0,2.774400,28358.0,0.197013,44.50
2,Hyderabad,540,8500000.0,2.072301,8200.0,0.049631,40.20
1,Chennai,1595,5870000.0,2.725424,58508.0,0.059771,36.20
0,Bangalore,4513,7800000.0,2.167500,28358.0,0.000000,32.50



Scoring complete. If results look odd, common next checks:
 - verify Locality matching with rent_grouped (alternate spellings/abbreviations).
 - inspect rows with rental_yield_pct == 0 to see why Avg_Rent was not found.
 - if Flood_Risk is missing/flat, consider enriching weather mapping by state.


In [210]:
# 1: show rows with zero rental yield to see why
df_zero = df[df["rental_yield_pct"] == 0].copy()
print("Count zero rental_yield_pct:", len(df_zero))
display(df_zero[[
    "Name","City","City_clean","Locality","Locality_clean",
    "Price","Avg_Rent","rental_yield_pct","ry_capped","ry_prank","investment_score"
]])


Count zero rental_yield_pct: 1


,Name,City,City_clean,Locality,Locality_clean,Price,Avg_Rent,rental_yield_pct,ry_capped,ry_prank,investment_score
4757,"Lake City, Bommanahalli,Bangalore","Bommanahalli,Bangalore",Bangalore,Lake City,Lake City,NaN,14450.0,0.0,0.0,0.000069,6.5


In [211]:
# find any columns that look like raw price fields
price_like = [c for c in df.columns if "price" in c.lower() and c != "Price"]
print("price-like cols found:", price_like)
# show the problematic row (index 4757 from your screenshot)
problem_idx = df_zero.index[0] if len(df_zero) else None
print("problem index:", problem_idx)
display(df.loc[[problem_idx], price_like + ["Price", "Avg_Rent", "Name", "City","Locality"]])


price-like cols found: ['valid_price']
problem index: 4757


,valid_price,Price,Avg_Rent,Name,City,Locality
4757,False,NaN,14450.0,"Lake City, Bommanahalli,Bangalore","Bommanahalli,Bangalore",Lake City


In [212]:
# assume you have the clean_price(x) function already defined in the notebook.
# Use first available price-like column to try to recover missing Price
if price_like:
    src = price_like[0]
    print("Attempting to create Price from", src)
    # convert raw values to cleaned numeric (non-destructive)
    recovered = df[src].apply(clean_price)
    # fill missing Price only where Price is NaN
    df["Price_recovered_from_raw"] = recovered
    before = df.loc[problem_idx, "Price"]
    df.loc[df["Price"].isna(), "Price"] = df.loc[df["Price"].isna(), "Price_recovered_from_raw"]
    after = df.loc[problem_idx, "Price"]
    print("Before:", before, "After:", after)
else:
    print("No raw price-like column found to attempt recovery.")


Attempting to create Price from valid_price
Before: nan After: nan


In [213]:
# compute city median price (skip NaNs)
city_median_price = df.groupby("City_clean")["Price"].median().to_dict()
# map into df (new column)
df["Price_from_city_median"] = df["City_clean"].map(city_median_price)
# fill Price only where still missing
df.loc[df["Price"].isna(), "Price"] = df.loc[df["Price"].isna(), "Price_from_city_median"]
# as final fallback use global median (should rarely be needed)
global_median = float(df["Price"].median(skipna=True))
df["Price"] = df["Price"].fillna(global_median)
print("Global median used (if any):", global_median)


Global median used (if any): 6500000.0


In [214]:
# recompute rental yield safely (mask Price>0)
df["Avg_Rent"] = pd.to_numeric(df.get("Avg_Rent", pd.Series(np.nan, index=df.index)), errors="coerce").fillna(0)
df["Price"] = pd.to_numeric(df.get("Price", pd.Series(np.nan, index=df.index)), errors="coerce")

mask = df["Price"] > 0
df["rental_yield_pct"] = 0.0
df.loc[mask, "rental_yield_pct"] = (df.loc[mask, "Avg_Rent"] * 12) / df.loc[mask, "Price"] * 100
df["rental_yield_pct"] = df["rental_yield_pct"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# reapply the 99th-percentile cap & percentile ranks (same logic as before)
cap_99 = df["rental_yield_pct"].quantile(0.99)
df["ry_capped"] = np.where(df["rental_yield_pct"] > cap_99, cap_99, df["rental_yield_pct"])
df["ry_prank"] = df["ry_capped"].rank(method="average", pct=True)

# recompute safety & investment_score (if you have Crime_Index and flood_norm present)
df["Crime_Index"] = pd.to_numeric(df.get("Crime_Index", pd.Series(np.nan, index=df.index)), errors="coerce").fillna(df["Crime_Index"].median())
df["crime_prank"] = df["Crime_Index"].rank(method="average", pct=True)
df["safety_prank"] = 1.0 - df["crime_prank"]

# Combine as before (70% rental, 30% safety)
df["investment_score"] = (0.7 * df["ry_prank"] + 0.3 * df["safety_prank"]) * 100
df["investment_score"] = df["investment_score"].round(1)

print("Recomputed rental_yield_pct and investment_score.")


Recomputed rental_yield_pct and investment_score.


In [215]:
display(df.loc[[problem_idx], [
    "Name","City","City_clean","Locality","Locality_clean",
    "Price_recovered_from_raw","Price_from_city_median","Price",
    "Avg_Rent","rental_yield_pct","ry_capped","ry_prank","investment_score"
]])
# quick summary counts
print("Count zero rental_yield_pct after fix:", int((df["rental_yield_pct"]==0).sum()))


,Name,City,City_clean,Locality,Locality_clean,Price_recovered_from_raw,Price_from_city_median,Price,Avg_Rent,rental_yield_pct,ry_capped,ry_prank,investment_score
4757,"Lake City, Bommanahalli,Bangalore","Bommanahalli,Bangalore",Bangalore,Lake City,Lake City,NaN,7800000.0,7800000.0,14450.0,2.223077,2.223077,0.3856,40.0


Count zero rental_yield_pct after fix: 0


In [216]:
# Safe repair + diagnostics (use this instead of the previous failing cell)
import pandas as pd
import numpy as np

# re-detect zero-yield rows
df_zero = df[df["rental_yield_pct"] == 0].copy()
print("Zero-yield rows found:", len(df_zero))

if len(df_zero) == 0:
    print("No zero rental_yield_pct rows found — nothing to display.")
    problem_idx = None
else:
    # pick first problem index (this is a label from the DataFrame index)
    problem_idx = df_zero.index[0]
    print("Problem index (first):", problem_idx)

# find price-like columns
price_like = [c for c in df.columns if ("price" in c.lower() and c not in ("Price", "price")) or c.lower().startswith("raw")]
print("price-like columns found:", price_like)

# show the problem row BEFORE fixes (if available)
if problem_idx is not None and problem_idx in df.index:
    cols_before = [c for c in price_like + ["Price", "Avg_Rent", "Name", "City", "Locality", "City_clean", "Locality_clean"] if c in df.columns]
    print("\nProblem row (BEFORE) — showing available columns:")
    display(df.loc[[problem_idx], cols_before])
else:
    print("\nSkipping BEFORE display (no problem index present or index not in df).")

# Try to recover price from first price-like column (if any) using clean_price
df["Price_recovered_from_raw"] = np.nan
if price_like and 'clean_price' in globals():
    src = price_like[0]
    try:
        recovered = df[src].apply(lambda x: clean_price(x) if pd.notna(x) else np.nan)
        df["Price_recovered_from_raw"] = pd.to_numeric(recovered, errors="coerce")
        # fill Price only where Price is missing
        mask_missing = df["Price"].isna()
        df.loc[mask_missing, "Price"] = df.loc[mask_missing, "Price_recovered_from_raw"]
        print(f"\nAttempted recovery from '{src}'. Filled {mask_missing.sum()} missing Price rows where available.")
    except Exception as e:
        print("Error applying clean_price:", e)
else:
    if not price_like:
        print("\nNo price-like raw column found to attempt recovery.")
    else:
        print("\nclean_price function not found — cannot recover from raw column.")

# Ensure Price numeric
df["Price"] = pd.to_numeric(df.get("Price", pd.Series(np.nan, index=df.index)), errors="coerce")

# Fill remaining Price using city median then global median
city_median_price = df.groupby("City_clean")["Price"].median().to_dict()
df["Price_from_city_median"] = df["City_clean"].map(city_median_price)

missing_before = int(df["Price"].isna().sum())
df.loc[df["Price"].isna(), "Price"] = df.loc[df["Price"].isna(), "Price_from_city_median"]
missing_after_city = int(df["Price"].isna().sum())
global_median = float(df["Price"].median(skipna=True)) if df["Price"].notna().any() else 0.0
df["Price"] = df["Price"].fillna(global_median)
missing_final = int(df["Price"].isna().sum())

print(f"\nPrice missing: before={missing_before}, after_city_median={missing_after_city}, final={missing_final}")
print("Global median Price used for final fill:", global_median)

# Recompute rental_yield_pct safely
df["Avg_Rent"] = pd.to_numeric(df.get("Avg_Rent", pd.Series(np.nan, index=df.index)), errors="coerce").fillna(0.0)
mask = df["Price"] > 0
df["rental_yield_pct"] = 0.0
df.loc[mask, "rental_yield_pct"] = (df.loc[mask, "Avg_Rent"] * 12) / df.loc[mask, "Price"] * 100
df["rental_yield_pct"] = df["rental_yield_pct"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# reapply cap and ranks
cap_99 = df["rental_yield_pct"].quantile(0.99)
df["ry_capped"] = np.where(df["rental_yield_pct"] > cap_99, cap_99, df["rental_yield_pct"])
df["ry_prank"] = df["ry_capped"].rank(method="average", pct=True)

df["Crime_Index"] = pd.to_numeric(df.get("Crime_Index", pd.Series(np.nan, index=df.index)), errors="coerce")
if df["Crime_Index"].notna().sum() == 0:
    df["Crime_Index"] = 0.0
else:
    df["Crime_Index"] = df["Crime_Index"].fillna(df["Crime_Index"].median())
df["crime_prank"] = df["Crime_Index"].rank(method="average", pct=True)
df["safety_prank"] = 1.0 - df["crime_prank"]

df["investment_score"] = (0.7 * df["ry_prank"] + 0.3 * df["safety_prank"]) * 100
df["investment_score"] = df["investment_score"].round(1)

# show the problem row AFTER repairs, if it exists
if problem_idx is not None and problem_idx in df.index:
    after_cols = [
        "Name","City","City_clean","Locality","Locality_clean",
        "Price_recovered_from_raw","Price_from_city_median","Price",
        "Avg_Rent","rental_yield_pct","ry_capped","ry_prank","investment_score"
    ]
    after_cols = [c for c in after_cols if c in df.columns]
    print("\nProblem row (AFTER) — showing available columns:")
    display(df.loc[[problem_idx], after_cols])
else:
    print("\nSkipping AFTER display (no problem index present or index not in df).")

# final diagnostics
print("\nFinal diagnostics:")
print("Total rows:", len(df))
print("Avg_Rent zeros:", int((df["Avg_Rent"] == 0).sum()))
print("rental_yield_pct zeros:", int((df["rental_yield_pct"] == 0).sum()))
print("Price still NaN:", int(df["Price"].isna().sum()))
print("99th percentile cap on rental_yield_pct:", cap_99)


Zero-yield rows found: 0
No zero rental_yield_pct rows found — nothing to display.
price-like columns found: ['valid_price', 'Price_recovered_from_raw', 'Price_from_city_median']

Skipping BEFORE display (no problem index present or index not in df).

Attempted recovery from 'valid_price'. Filled 0 missing Price rows where available.

Price missing: before=0, after_city_median=0, final=0
Global median Price used for final fill: 6500000.0

Skipping AFTER display (no problem index present or index not in df).

Final diagnostics:
Total rows: 14528
Avg_Rent zeros: 0
rental_yield_pct zeros: 0
Price still NaN: 0
99th percentile cap on rental_yield_pct: 26.39852413242901


In [217]:
# save cleaned master CSV
df.to_csv("../data/master_dataset_final.csv", index=False)
print("Saved ../data/master_dataset_final.csv")


Saved ../data/master_dataset_final.csv


In [ ]:
df["score_driver"] = np.select(
    [
        df["score_rent_component"] >= df[["score_safety_component","score_flood_component"]].max(axis=1),
        df["score_safety_component"] >= df[["score_rent_component","score_flood_component"]].max(axis=1),
    ],
    [
        "Rental Yield Driven",
        "Safety Driven"
    ],
    default="Mixed Factors"
)


In [ ]:
import numpy as np
import pandas as pd

# Convert Avg_Rent to numeric
df["Avg_Rent"] = pd.to_numeric(df["Avg_Rent"], errors="coerce")

# Treat 0 as missing
df.loc[df["Avg_Rent"] == 0, "Avg_Rent"] = np.nan

# Fill using City median
df["Avg_Rent"] = df["Avg_Rent"].fillna(
    df.groupby("City_clean")["Avg_Rent"].transform("median")
)

# Final fallback: global median
df["Avg_Rent"] = df["Avg_Rent"].fillna(df["Avg_Rent"].median())


In [ ]:
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

df["rental_yield_pct"] = 0.0
mask = df["Price"] > 0

df.loc[mask, "rental_yield_pct"] = (
    df.loc[mask, "Avg_Rent"] * 12
) / df.loc[mask, "Price"] * 100

df["rental_yield_pct"] = (
    df["rental_yield_pct"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)


In [222]:
import pandas as pd

base = pd.read_csv("../data/master_dataset_final.csv")          # raw dataset
frozen = pd.read_csv("../data/master_dataset_FINAL_FROZEN_v3.csv")  # frozen dataset


In [234]:
import uuid

base["property_id"] = [str(uuid.uuid4()) for _ in range(len(base))]


In [235]:
base.to_csv("master_dataset_final_WITH_ID.csv", index=False)


In [242]:
base["join_key"] = (
    base["City_clean"].astype(str) + "|" +
    base["Locality"].astype(str) + "|" +
    base["Price"].astype(str)
)

frozen["join_key"] = (
    frozen["City_clean"].astype(str) + "|" +
    frozen["Locality"].astype(str) + "|" +
    frozen["Price"].astype(str)
)


In [244]:
base_reduced = (
    base
    .groupby("join_key", as_index=False)
    .agg({
        "Name": "first",
        "Property Title": "first",
        "State": "first",
        "Area_sqft": "first",
        "ry_prank": "mean",
        "ry_capped": "mean",
        "crime_prank": "mean",
        "safety_prank": "mean",
        "flood_norm": "mean"
    })
)


In [245]:
frozen_final = frozen.merge(
    base_reduced,
    on="join_key",
    how="left"
)


In [246]:
print("Duplicate property_id:",
      frozen_final["property_id"].duplicated().sum())

print("Missing crime_prank:",
      frozen_final["crime_prank"].isna().sum())


Duplicate property_id: 0
Missing crime_prank: 0


In [247]:
frozen_final.drop(columns=["join_key"], inplace=True)

frozen_final.to_csv(
    "../data/master_dataset_FINAL_FROZEN_READY.csv",
    index=False
)

print("✅ FINAL DATASET FROZEN — USE THIS FILE ONLY")


✅ FINAL DATASET FROZEN — USE THIS FILE ONLY
